# Evaluación Comparativa Final — IPC Global

**Pregunta central**: ¿Mejoran las señales institucionales globales (Fed Funds, Brent, EPU) la predicción del IPC global en foundation models frente a los modelos estadísticos clásicos?

Backtesting rolling-origin 2021-2024 · Horizontes h=1,3,6,12 meses

**Modelos disponibles:**
- **Baseline estadístico**: Naive, ARIMA, ARIMA(1,1,1), ARIMAX (con Fed Funds), ARIMAX C1_inst
- **Deep Learning**: LSTM, N-BEATS, N-HiTS  *(ventana de evaluación más corta: ~16 orígenes)*
- **Foundation (C1_inst)**: Chronos-2, TimesFM, TimeGPT — solo con señales institucionales (sin MCP)

> **Nota**: Para el IPC Global no se dispone de señales MCP (BCE/GDELT), por lo que el experimento solo incluye la condición C1_inst.

**Secciones:**
1. ¿Qué modelo predice mejor en cada horizonte?
2. ¿Los foundation models superan a ARIMA? (Δ MAE relativo)
3. ¿Hay diferencias por periodo (pre-crisis / shock / normalización)?
4. ¿Son las diferencias estadísticamente significativas (test Diebold-Mariano)?
5. **Figura principal** para la memoria del TFG

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import warnings
warnings.filterwarnings('ignore')

ROOT    = Path('..').resolve()
RESULTS = ROOT / '08_results'
HORIZONS = [1, 3, 6, 12]

# ── Cargar métricas desde ficheros individuales ────────────────────
baseline_raw  = json.load(open(RESULTS / 'rolling_metrics_global.json'))
baseline_c1   = json.load(open(RESULTS / 'rolling_metrics_C1_inst_global.json'))
deep_raw      = json.load(open(RESULTS / 'deep_rolling_metrics_global.json'))

FOUNDATION_MODELS_GLOBAL = [
    'chronos2_C1_inst_global',
    'timesfm_C1_inst_global',
    'timegpt_C1_inst_global',
]

metrics = {}
for k, v in baseline_raw.items():
    metrics[k] = v
for k, v in baseline_c1.items():
    metrics[k] = v
for k, v in deep_raw.items():
    metrics[k] = v
for name in FOUNDATION_MODELS_GLOBAL:
    p = RESULTS / f'{name}_metrics.json'
    if p.exists():
        d = json.load(open(p))
        metrics[name] = d[name]

# Escala MASE: naive h=1 MAE / naive h=1 MASE
MASE_SCALE = round(baseline_raw['naive']['h1']['MAE'] / baseline_raw['naive']['h1']['MASE'], 4)

print(f'Modelos cargados: {len(metrics)}')
print(f'MASE scale: {MASE_SCALE:.4f}')
print('Modelos:', list(metrics.keys()))

In [ ]:
# ── Paleta y orden ─────────────────────────────────────────────────
FAMILY_COLORS = {
    'naive':              '#b0b0b0',
    'arima':              '#606060',
    'arima111':           '#404040',
    'arimax':             '#202020',
    'auto_arima':         '#8B4513',
    'arimax_C1_inst':     '#333366',
    'lstm':               '#4e9af1',
    'nbeats':             '#1a6ab0',
    'nhits':              '#0d3d6e',
    'chronos2_C1_inst_global': '#52a852',
    'timesfm_C1_inst_global':  '#9467bd',
    'timegpt_C1_inst_global':  '#8fbc8f',
}

MODEL_ORDER = [
    'naive', 'arima', 'arima111', 'arimax', 'auto_arima', 'arimax_C1_inst',
    'lstm', 'nbeats', 'nhits',
    'chronos2_C1_inst_global',
    'timesfm_C1_inst_global',
    'timegpt_C1_inst_global',
]
MODEL_ORDER = [m for m in MODEL_ORDER if m in metrics]

LABELS = {
    'naive':              'Naive (lag-12)',
    'arima':              'ARIMA (auto)',
    'arima111':           'ARIMA(1,1,1)',
    'arimax':             'ARIMAX (Fed Funds)',
    'auto_arima':         'AutoARIMA (dinámico) ★',
    'arimax_C1_inst':     'ARIMAX C1_inst',
    'lstm':               'LSTM',
    'nbeats':             'N-BEATS',
    'nhits':              'N-HiTS',
    'chronos2_C1_inst_global': 'Chronos-2 C1_inst ★★',
    'timesfm_C1_inst_global':  'TimesFM C1_inst',
    'timegpt_C1_inst_global':  'TimeGPT C1_inst',
}

# ── DataFrame plano de métricas ────────────────────────────────────
rows = []
for model in MODEL_ORDER:
    for h in HORIZONS:
        m = metrics[model].get(f'h{h}', {})
        rows.append({'model': model, 'horizon': h,
                     'MAE': m.get('MAE'), 'RMSE': m.get('RMSE'),
                     'MASE': m.get('MASE'), 'n': m.get('n_evals')})
df = pd.DataFrame(rows)

print('Tabla resumen MAE:')
piv = df.pivot(index='model', columns='horizon', values='MAE').reindex(MODEL_ORDER)
piv.columns = [f'h={h}' for h in piv.columns]
piv.index = [LABELS.get(m, m) for m in piv.index]
print(piv.round(4).to_string())

---
## 1 · ¿Qué modelo predice mejor el IPC Global en cada horizonte?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 6), sharey=False)

# Ordenar por MAE h=1 (sin naive)
sorted_mods = (df[(df['horizon']==1) & (df['model']!='naive')]
               .set_index('model')['MAE']
               .reindex([m for m in MODEL_ORDER if m != 'naive'])
               .dropna().sort_values().index.tolist())

for ax, h in zip(axes, HORIZONS):
    sub = (df[(df['horizon']==h) & (df['model'].isin(sorted_mods))]
           .set_index('model').reindex(sorted_mods))
    colors = [FAMILY_COLORS.get(m, '#999') for m in sorted_mods]
    ax.barh(range(len(sorted_mods)), sub['MAE'], color=colors,
            edgecolor='white', lw=0.4)
    best_idx = sub['MAE'].dropna().idxmin()
    ax.barh(sorted_mods.index(best_idx), sub.loc[best_idx,'MAE'],
            color=FAMILY_COLORS.get(best_idx,'#999'), edgecolor='gold', lw=2.5)
    ax.set_yticks(range(len(sorted_mods)))
    ax.set_yticklabels([LABELS.get(m, m) for m in sorted_mods], fontsize=8.5)
    ax.set_xlabel('MAE (pp IPC)', fontsize=9)
    ax.set_title(f'h={h}', fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.invert_yaxis()

legend_items = [
    mpatches.Patch(color='#404040', label='Estadístico (ARIMA/ARIMAX)'),
    mpatches.Patch(color='#333366', label='ARIMAX C1_inst (Fed+EPU+Brent)'),
    mpatches.Patch(color='#1a6ab0', label='Deep Learning (LSTM/N-BEATS/N-HiTS)'),
    mpatches.Patch(color='#52a852', label='Foundation C1_inst ★ — Chronos-2 (mejor)'),
    mpatches.Patch(color='#9467bd', label='Foundation C1_inst — TimesFM'),
    mpatches.Patch(color='#8fbc8f', label='Foundation C1_inst — TimeGPT'),
]
fig.legend(handles=legend_items, loc='lower center', ncol=3, fontsize=8.5,
           frameon=True, bbox_to_anchor=(0.5, -0.07))
fig.suptitle('MAE por modelo y horizonte — IPC Global · Rolling-origin 2021-2024\n'
             '(borde dorado = mejor de ese horizonte | naive excluido del ranking)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(RESULTS / 'fig1_mae_all_models_global.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop-5 por horizonte:')
for h in HORIZONS:
    top = df[(df['horizon']==h) & (df['model']!='naive')].sort_values('MAE').head(5)
    print(f'h={h}: ' + ' | '.join(
        f"{LABELS.get(r['model'],r['model']):30s} {r['MAE']:.4f}" for _, r in top.iterrows()))

---
## 2 · ¿Los foundation models superan a ARIMA? (Δ MAE relativo)

Para el IPC Global solo hay condición C1_inst (sin señales MCP). La referencia natural es ARIMA.

In [ ]:
# ── Δ MAE relativo: foundation C1_inst vs ARIMA ────────────────────
DELTA_PAIRS = [
    ('AutoARIMA (dinámico) ★', 'arima',  'auto_arima'),
    ('ARIMAX C1_inst',          'arima',  'arimax_C1_inst'),
    ('Chronos-2 C1_inst ★★',   'arima',  'chronos2_C1_inst_global'),
    ('TimesFM C1_inst',         'arima',  'timesfm_C1_inst_global'),
    ('TimeGPT C1_inst',         'arima',  'timegpt_C1_inst_global'),
    ('LSTM',                    'arima',  'lstm'),
    ('N-BEATS',                 'arima',  'nbeats'),
    ('N-HiTS',                  'arima',  'nhits'),
]

hm = np.full((len(DELTA_PAIRS), 4), np.nan)
for i, (lbl, ref, m2) in enumerate(DELTA_PAIRS):
    for j, h in enumerate(HORIZONS):
        m_ref = metrics.get(ref, {}).get(f'h{h}', {}).get('MAE')
        m2v   = metrics.get(m2,  {}).get(f'h{h}', {}).get('MAE')
        if m_ref and m2v:
            hm[i, j] = (m2v - m_ref) / m_ref * 100

fig, ax = plt.subplots(figsize=(8, 7))
norm = TwoSlopeNorm(vmin=-30, vcenter=0, vmax=80)
im = ax.imshow(hm, cmap='RdYlGn_r', norm=norm, aspect='auto')
ax.set_xticks(range(4))
ax.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=11)
ax.set_yticks(range(len(DELTA_PAIRS)))
ax.set_yticklabels([r[0] for r in DELTA_PAIRS], fontsize=10)
ax.set_title('Δ MAE relativo: (modelo − ARIMA) / ARIMA × 100%\n'
             'Verde = mejor que ARIMA  |  Rojo = peor que ARIMA',
             fontsize=10, fontweight='bold')
for i in range(len(DELTA_PAIRS)):
    for j in range(4):
        v = hm[i, j]
        if not np.isnan(v):
            ct = 'white' if abs(v) > 30 else 'black'
            ax.text(j, i, f'{v:+.1f}%', ha='center', va='center',
                    fontsize=9.5, color=ct, fontweight='bold')
# Separadores: AutoARIMA | estadístico+DL | foundation
ax.axhline(0.5, color='white', lw=2.5)
ax.axhline(1.5, color='white', lw=2.0)
ax.axhline(4.5, color='white', lw=2.0)
plt.colorbar(im, ax=ax, shrink=0.75, label='Δ MAE (%)')
plt.tight_layout()
plt.savefig(RESULTS / 'fig2_delta_mae_heatmap_global.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Perfiles MAE: comparativa modelos clave ────────────────────────
PROFILE_MODS = [
    ('arima',                  '#606060', 'ARIMA (referencia)',         'o-',  2.0),
    ('arimax',                 '#404040', 'ARIMAX (Fed Funds)',         's:',  1.6),
    ('arimax_C1_inst',         '#333366', 'ARIMAX C1_inst',             'D-',  1.6),
    ('auto_arima',             '#8B4513', 'AutoARIMA (dinámico) ★',     'x--', 2.0),
    ('nbeats',                 '#1a6ab0', 'Deep: N-BEATS',              '^--', 1.8),
    ('chronos2_C1_inst_global','#52a852', 'Chronos-2 C1_inst ★★',      'P-',  2.4),
    ('timesfm_C1_inst_global', '#9467bd', 'TimesFM C1_inst',            'v--', 2.0),
    ('timegpt_C1_inst_global', '#8fbc8f', 'TimeGPT C1_inst',            'h:',  1.8),
]

fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(HORIZONS))

for mn, col, lbl, sty, lw in PROFILE_MODS:
    if mn not in metrics: continue
    vals = [metrics[mn].get(f'h{h}', {}).get('MAE') for h in HORIZONS]
    if all(v is not None for v in vals):
        ax.plot(x, vals, sty, color=col, lw=lw, ms=8, label=lbl, zorder=5)

ax.set_xticks(x)
ax.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=11)
ax.set_ylabel('MAE (pp IPC Global)', fontsize=10)
ax.set_xlabel('Horizonte de predicción (meses)', fontsize=10)
ax.set_title('Perfil MAE — IPC Global · Baseline vs AutoARIMA vs Foundation\n'
             '★ AutoARIMA dinámico mejora ARIMA fijo (-6% h=1, -14% h=12)\n'
             '★★ Chronos-2 C1_inst es el mejor modelo global',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9, loc='upper left', framealpha=0.88)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(RESULTS / 'fig3_profiles_global.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3 · Error temporal mes a mes — ¿Cuándo falla cada modelo?

Error absoluto h=1 a lo largo de 2021-2024. Fondos sombreados:
**A** — pre-crisis (2021-01 a 2022-06), **B** — shock inflacionista (2022-07 a 2023-06), **C** — normalización (2023-07 a 2024-12).

In [ ]:
BASELINE_PREDS = RESULTS / 'rolling_predictions_global.parquet'
DEEP_PREDS     = RESULTS / 'deep_rolling_predictions_global.parquet'

_df_base = pd.read_parquet(BASELINE_PREDS)
_df_base['origin']  = pd.to_datetime(_df_base['origin'])
_df_base['fc_date'] = pd.to_datetime(_df_base['fc_date'])

_df_deep = pd.read_parquet(DEEP_PREDS)
_df_deep['origin']  = pd.to_datetime(_df_deep['origin'])
_df_deep['fc_date'] = pd.to_datetime(_df_deep['fc_date'])

print('Modelos en baseline parquet:', _df_base['model'].unique().tolist())
print('Modelos en deep parquet:', _df_deep['model'].unique().tolist())

In [ ]:
def _load_foundation_global(name):
    d = pd.read_parquet(RESULTS / f'{name}_predictions.parquet')
    d['origin']  = pd.to_datetime(d['origin'])
    d['fc_date'] = pd.to_datetime(d['fc_date'])
    return d

def _get_h1_errors(name):
    if name in ('lstm', 'nbeats', 'nhits'):
        sub = _df_deep[(_df_deep['model']==name) & (_df_deep['horizon']==1) & (_df_deep['step']==1)]
    elif name in ('arima', 'arima111', 'arimax', 'naive'):
        sub = _df_base[(_df_base['model']==name) & (_df_base['horizon']==1) & (_df_base['step']==1)]
    else:
        d = _load_foundation_global(name)
        sub = d[(d['horizon']==1) & (d['step']==1)]
    return sub.set_index('fc_date')['abs_error'].sort_index()

SHADING = [
    ('A: Pre-crisis',   '2021-01-01', '2022-06-30', '#d4e8d4', 0.45),
    ('B: Shock',        '2022-07-01', '2023-06-30', '#f8d7d7', 0.50),
    ('C: Normalización','2023-07-01', '2024-12-31', '#d7e8f8', 0.35),
]

TIMELINE_MODELS = [
    ('arima',                  '#606060', 'ARIMA',                '-',  1.8, 'o'),
    ('arimax_C1_inst',         '#333366', 'ARIMAX C1_inst',       ':',  1.5, 's'),
    ('nbeats',                 '#1a6ab0', 'N-BEATS',              '--', 1.8, 'D'),
    ('chronos2_C1_inst_global','#52a852', 'Chronos-2 C1_inst ★',  '-',  2.2, '^'),
    ('timesfm_C1_inst_global', '#9467bd', 'TimesFM C1_inst',      '--', 1.8, 'v'),
    ('timegpt_C1_inst_global', '#8fbc8f', 'TimeGPT C1_inst',      ':',  1.5, 'h'),
]

fig, ax = plt.subplots(figsize=(14, 5))
for lbl, s, e, col, alpha in SHADING:
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=col, alpha=alpha, zorder=0)

for name, col, label, ls, lw, mk in TIMELINE_MODELS:
    try:
        series = _get_h1_errors(name)
        if len(series) == 0:
            print(f'[!] {name}: sin datos')
            continue
        ax.plot(series.index, series.values, ls=ls, color=col, lw=lw,
                marker=mk, markersize=4, markevery=4, label=label, zorder=5)
    except Exception as ex:
        print(f'[!] {name}: {ex}')

ylim = ax.get_ylim()
for lbl, s, e, col, alpha in SHADING:
    ax.text(pd.Timestamp(s) + pd.Timedelta(days=22), ylim[1]*0.96,
            lbl, fontsize=8.5, color='#333', va='top', fontweight='bold')

ax.set_ylabel('Error absoluto h=1 (pp IPC)', fontsize=10)
ax.set_xlabel('Fecha de predicción', fontsize=10)
ax.set_title('Error absoluto mes a mes — h=1 · IPC Global 2021-2024', fontsize=11, fontweight='bold')
ax.legend(fontsize=9, loc='upper left', framealpha=0.85)
ax.grid(alpha=0.22, zorder=0)
ax.set_xlim(pd.Timestamp('2021-01-01'), pd.Timestamp('2025-01-01'))
plt.tight_layout()
plt.savefig(RESULTS / 'fig3b_error_temporal_global.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4 · Test Diebold-Mariano

Se calculan los DM tests directamente desde las predicciones parquet, comparando cada foundation model (y ARIMAX C1_inst) contra ARIMA como referencia.

In [ ]:
def harvey_dm_test(e1, e2, h=1, power=1):
    """Harvey, Leybourne & Newbold (1997) DM test. DM<0 → e1 better."""
    d = np.abs(e1)**power - np.abs(e2)**power
    T = len(d)
    if T < 4:
        return np.nan, np.nan
    d_bar = np.mean(d)
    # HAC variance with autocovariances up to lag h-1
    gamma0 = np.var(d, ddof=1)
    acs = [np.cov(d[k:], d[:-k])[0,1] if k > 0 else gamma0 for k in range(h)]
    V_d = (gamma0 + 2 * sum(acs[1:])) / T
    if V_d <= 0:
        return np.nan, np.nan
    dm = d_bar / np.sqrt(V_d)
    # Harvey et al. small-sample correction
    dm_adj = dm * np.sqrt((T + 1 - 2*h + h*(h-1)/T) / T)
    p = 2 * stats.t.sf(np.abs(dm_adj), df=T-1)
    return float(dm_adj), float(p)


def load_preds_for_dm(name):
    """Devuelve DataFrame con columnas: origin, fc_date, step, horizon, y_true, y_pred"""
    if name in ('arima', 'arima111', 'arimax', 'naive'):
        sub = _df_base[_df_base['model']==name].copy()
    elif name == 'arimax_C1_inst':
        d = pd.read_parquet(RESULTS / 'rolling_predictions_C1_inst_global.parquet')
        d['origin']  = pd.to_datetime(d['origin'])
        d['fc_date'] = pd.to_datetime(d['fc_date'])
        sub = d[d['model']=='arimax_C1_inst'].copy()
    elif name in ('lstm', 'nbeats', 'nhits'):
        sub = _df_deep[_df_deep['model']==name].copy()
    else:
        sub = _load_foundation_global(name).copy()
    return sub


SUBPERIODS = {
    'global':          (pd.Timestamp('2021-01-01'), pd.Timestamp('2024-12-31')),
    'A_pre_crisis':    (pd.Timestamp('2021-01-01'), pd.Timestamp('2022-06-30')),
    'B_shock':         (pd.Timestamp('2022-07-01'), pd.Timestamp('2023-06-30')),
    'C_normalizacion': (pd.Timestamp('2023-07-01'), pd.Timestamp('2024-12-31')),
}

DM_PAIRS = [
    ('arima', 'arimax_C1_inst'),
    ('arima', 'chronos2_C1_inst_global'),
    ('arima', 'timesfm_C1_inst_global'),
    ('arima', 'timegpt_C1_inst_global'),
    ('chronos2_C1_inst_global', 'timesfm_C1_inst_global'),
    ('chronos2_C1_inst_global', 'timegpt_C1_inst_global'),
]

dm_results = []

for m1_name, m2_name in DM_PAIRS:
    try:
        p1 = load_preds_for_dm(m1_name)
        p2 = load_preds_for_dm(m2_name)
    except Exception as ex:
        print(f'[!] No se pueden cargar {m1_name}/{m2_name}: {ex}')
        continue

    for period, (t_start, t_end) in SUBPERIODS.items():
        for h in HORIZONS:
            key = {'model1': m1_name, 'model2': m2_name, 'period': period, 'horizon': h}

            e1 = (p1[(p1['horizon']==h) & (p1['fc_date']>=t_start) & (p1['fc_date']<=t_end)]
                  .sort_values(['origin','step'])['error'].values)
            e2 = (p2[(p2['horizon']==h) & (p2['fc_date']>=t_start) & (p2['fc_date']<=t_end)]
                  .sort_values(['origin','step'])['error'].values)

            n_common = min(len(e1), len(e2))
            if n_common < 10:
                dm_results.append({**key, 'dm_stat': np.nan, 'p_value': np.nan,
                                   'better': 'insuf', 'n': n_common})
                continue

            e1, e2 = e1[:n_common], e2[:n_common]
            dm_stat, p_val = harvey_dm_test(e1, e2, h=h, power=1)
            if pd.isna(dm_stat):
                better = 'insuf'
            elif p_val < 0.05:
                better = 'model1' if dm_stat < 0 else 'model2'
            else:
                better = 'tie'
            dm_results.append({**key, 'dm_stat': dm_stat, 'p_value': p_val,
                               'better': better, 'n': n_common})

dm_df = pd.DataFrame(dm_results)
print(f'Tests DM calculados: {len(dm_df)}')

# Guardar para referencia
import json
dm_out = []
for (m1, m2, per), grp in dm_df.groupby(['model1','model2','period']):
    entry = {'model1': m1, 'model2': m2, 'period': per}
    for _, row in grp.iterrows():
        entry[f'h{row["horizon"]}'] = {
            'dm_stat': round(row['dm_stat'], 4) if pd.notna(row['dm_stat']) else None,
            'p_value': round(row['p_value'], 4) if pd.notna(row['p_value']) else None,
            'better': row['better'], 'n': int(row['n'])
        }
    dm_out.append(entry)

json.dump(dm_out, open(RESULTS / 'diebold_mariano_results_global.json', 'w'), indent=2)
print('Guardado: diebold_mariano_results_global.json')

In [ ]:
# ── DM heatmap global: foundation vs ARIMA ────────────────────────
DM_PLOT_PAIRS = [
    ('arima', 'arimax_C1_inst',          'ARIMAX C1_inst vs ARIMA'),
    ('arima', 'chronos2_C1_inst_global', 'Chronos-2 vs ARIMA'),
    ('arima', 'timesfm_C1_inst_global',  'TimesFM vs ARIMA'),
    ('arima', 'timegpt_C1_inst_global',  'TimeGPT vs ARIMA'),
    ('chronos2_C1_inst_global', 'timesfm_C1_inst_global',  'Chronos-2 vs TimesFM'),
    ('chronos2_C1_inst_global', 'timegpt_C1_inst_global',  'Chronos-2 vs TimeGPT'),
]

dm_hm = np.full((len(DM_PLOT_PAIRS), 4), np.nan)
dm_pv = np.full((len(DM_PLOT_PAIRS), 4), np.nan)

for i, (m1, m2, lbl) in enumerate(DM_PLOT_PAIRS):
    sub = dm_df[(dm_df['model1']==m1) & (dm_df['model2']==m2) & (dm_df['period']=='global')]
    for j, h in enumerate(HORIZONS):
        hr = sub[sub['horizon']==h]
        if len(hr) and pd.notna(hr['dm_stat'].values[0]):
            dm_hm[i, j] = float(hr['dm_stat'].values[0])
            dm_pv[i, j] = float(hr['p_value'].values[0])

fig, ax = plt.subplots(figsize=(8, 6))
norm2 = TwoSlopeNorm(vmin=-5, vcenter=0, vmax=5)
im2 = ax.imshow(dm_hm, cmap='RdYlGn', norm=norm2, aspect='auto')
ax.set_xticks(range(4))
ax.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=10)
ax.set_yticks(range(len(DM_PLOT_PAIRS)))
ax.set_yticklabels([r[2] for r in DM_PLOT_PAIRS], fontsize=9.5)
ax.set_title('Estadístico DM — período completo (2021-2024)\n'
             'Verde = model2 mejor  |  Rojo = model1 mejor  |  ** p<0.05  * p<0.10\n'
             'Para primeras 4 filas: DM>0 → foundation/arimax mejor que ARIMA',
             fontsize=9.5, fontweight='bold')
for i in range(len(DM_PLOT_PAIRS)):
    for j in range(4):
        if not np.isnan(dm_hm[i, j]):
            sig = '**' if dm_pv[i,j] < 0.05 else ('*' if dm_pv[i,j] < 0.10 else '')
            ct  = 'white' if abs(dm_hm[i,j]) > 2.8 else 'black'
            ax.text(j, i, f'{dm_hm[i,j]:.2f}{sig}', ha='center', va='center',
                    fontsize=9, color=ct)
ax.axhline(3.5, color='white', lw=2.5)
plt.colorbar(im2, ax=ax, shrink=0.75, label='DM statistic')
plt.tight_layout()
plt.savefig(RESULTS / 'fig4_dm_global_heatmap_global.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── DM por subperiodo: ARIMA vs Chronos-2 C1_inst ─────────────────
PERIODS_SP = ['global', 'A_pre_crisis', 'B_shock', 'C_normalizacion']
PERIOD_LABELS = {
    'global':          'Global\n2021-2024',
    'A_pre_crisis':    'A: Pre-crisis\n2021-06/22',
    'B_shock':         'B: Shock\n2022-07/23',
    'C_normalizacion': 'C: Normalización\n2023-07/24',
}

PAIRS_SUB = [
    ('arima', 'chronos2_C1_inst_global', 'ARIMA vs Chronos-2 C1_inst'),
    ('arima', 'timesfm_C1_inst_global',  'ARIMA vs TimesFM C1_inst'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, (m1, m2, title) in zip(axes, PAIRS_SUB):
    sub = dm_df[(dm_df['model1']==m1) & (dm_df['model2']==m2)]
    xp = np.arange(len(PERIODS_SP))
    w  = 0.18
    for i, h in enumerate(HORIZONS):
        dvals, pvals = [], []
        for p in PERIODS_SP:
            row = sub[(sub['period']==p) & (sub['horizon']==h)]
            if len(row) and pd.notna(row['dm_stat'].values[0]):
                dvals.append(float(row['dm_stat'].values[0]))
                pvals.append(float(row['p_value'].values[0]))
            else:
                dvals.append(0.0); pvals.append(1.0)
        ax.bar(xp + (i-1.5)*w, dvals, w, label=f'h={h}',
               alpha=0.85, color=plt.cm.plasma(i/4))
        for xi, (dv, pv) in enumerate(zip(dvals, pvals)):
            if pv < 0.05:
                ax.text(xi+(i-1.5)*w, dv+(0.18 if dv>=0 else -0.42),
                        '**', ha='center', fontsize=9, fontweight='bold')
            elif pv < 0.10:
                ax.text(xi+(i-1.5)*w, dv+(0.18 if dv>=0 else -0.38),
                        '*', ha='center', fontsize=9)
    ax.axhline(0,     color='black', lw=1)
    ax.axhline(-1.96, color='red',   lw=0.9, ls='--', alpha=0.55, label='α=5% (m1 mejor)')
    ax.axhline( 1.96, color='green', lw=0.9, ls='--', alpha=0.55, label='α=5% (m2 mejor)')
    ax.set_xticks(xp)
    ax.set_xticklabels([PERIOD_LABELS[p] for p in PERIODS_SP], fontsize=9)
    ax.set_ylabel('Estadístico DM', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.legend(fontsize=7.5, loc='lower right', ncol=2)
    ax.grid(axis='y', alpha=0.25)
    ax.set_ylim(-7, 7)
    ax.annotate('DM > 0 → model2 mejor\nDM < 0 → ARIMA mejor',
                xy=(0.02, 0.97), xycoords='axes fraction', fontsize=7,
                va='top', color='gray', bbox=dict(boxstyle='round', fc='white', alpha=0.7))

fig.suptitle('Test Diebold-Mariano por subperiodo — ARIMA vs Foundation · IPC Global\n'
             '** p<0.05  * p<0.10', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'fig4b_dm_subperiod_global.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5 · Figura Principal — Para la Memoria del TFG

In [ ]:
matplotlib.rcParams.update({'font.family': 'DejaVu Sans',
                             'axes.spines.top': False, 'axes.spines.right': False})

fig = plt.figure(figsize=(17, 12))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.44, wspace=0.34)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[1, 0])
ax_d = fig.add_subplot(gs[1, 1])

# ── A: Ranking MAE h=12 (sin naive) ──────────────────────────────
df_h12 = (df[(df['horizon']==12) & (df['model']!='naive')]
           .dropna(subset=['MAE']).sort_values('MAE'))
mods_a  = df_h12['model'].tolist()
cols_a  = [FAMILY_COLORS.get(m, '#999') for m in mods_a]
ax_a.barh(range(len(mods_a)), df_h12['MAE'].values, color=cols_a,
           edgecolor='white', lw=0.5)
ax_a.barh(0, df_h12['MAE'].iloc[0], color=cols_a[0], edgecolor='gold', lw=2.5)
ax_a.set_yticks(range(len(mods_a)))
ax_a.set_yticklabels([LABELS.get(m, m) for m in mods_a], fontsize=8.5)
ax_a.set_xlabel('MAE (pp IPC)', fontsize=9)
ax_a.set_title('(A) Ranking MAE — h=12', fontsize=10, fontweight='bold')
ax_a.grid(axis='x', alpha=0.22)
ax_a.invert_yaxis()

# ── B: Perfiles MAE (con AutoARIMA) ───────────────────────────────
PROFILE_B = [
    ('arima',                  '#606060', 'ARIMA (referencia)',     'o-',  2.0),
    ('arimax',                 '#404040', 'ARIMAX (Fed Funds)',     's:',  1.5),
    ('auto_arima',             '#8B4513', 'AutoARIMA (dinámico) ★', 'x--', 2.0),
    ('nbeats',                 '#1a6ab0', 'Deep: N-BEATS',          '^--', 1.8),
    ('chronos2_C1_inst_global','#52a852', 'Chronos-2 C1_inst ★★',  'P-',  2.4),
    ('timesfm_C1_inst_global', '#9467bd', 'TimesFM C1_inst',        'v--', 2.0),
    ('timegpt_C1_inst_global', '#8fbc8f', 'TimeGPT C1_inst',        'h:',  1.5),
]
xb = np.arange(len(HORIZONS))
for mn, col, lbl, sty, lw in PROFILE_B:
    if mn not in metrics: continue
    vals = [metrics[mn].get(f'h{h}', {}).get('MAE') for h in HORIZONS]
    if all(v is not None for v in vals):
        ax_b.plot(xb, vals, sty, color=col, lw=lw, ms=7, label=lbl, zorder=5)
ax_b.set_xticks(xb)
ax_b.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=9)
ax_b.set_ylabel('MAE (pp IPC Global)', fontsize=9)
ax_b.set_title('(B) Perfil MAE — Modelos clave', fontsize=10, fontweight='bold')
ax_b.legend(fontsize=7.5, loc='upper left', framealpha=0.75)
ax_b.grid(alpha=0.22)

# ── C: Heatmap Δ MAE vs ARIMA (con AutoARIMA) ─────────────────────
HM_C = [
    ('AutoARIMA (dinámico) ★',  'arima', 'auto_arima'),
    ('ARIMAX C1_inst',           'arima', 'arimax_C1_inst'),
    ('Chronos-2 C1_inst ★★',    'arima', 'chronos2_C1_inst_global'),
    ('TimesFM C1_inst',          'arima', 'timesfm_C1_inst_global'),
    ('TimeGPT C1_inst',          'arima', 'timegpt_C1_inst_global'),
    ('LSTM',                     'arima', 'lstm'),
    ('N-BEATS',                  'arima', 'nbeats'),
    ('N-HiTS',                   'arima', 'nhits'),
]
hmc = np.full((len(HM_C), 4), np.nan)
for i, (_, ref, m2) in enumerate(HM_C):
    for j, h in enumerate(HORIZONS):
        mr = metrics.get(ref, {}).get(f'h{h}', {}).get('MAE')
        m2v= metrics.get(m2,  {}).get(f'h{h}', {}).get('MAE')
        if mr and m2v: hmc[i, j] = (m2v - mr) / mr * 100

norm_c = TwoSlopeNorm(vmin=-30, vcenter=0, vmax=80)
imc = ax_c.imshow(hmc, cmap='RdYlGn_r', norm=norm_c, aspect='auto')
ax_c.set_xticks(range(4))
ax_c.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=9)
ax_c.set_yticks(range(len(HM_C)))
ax_c.set_yticklabels([r[0] for r in HM_C], fontsize=9)
ax_c.set_title('(C) Δ MAE vs ARIMA\nVerde=mejora · Rojo=empeora',
               fontsize=10, fontweight='bold')
for i in range(len(HM_C)):
    for j in range(4):
        v = hmc[i, j]
        if not np.isnan(v):
            ct = 'white' if abs(v) > 30 else 'black'
            ax_c.text(j, i, f'{v:+.1f}%', ha='center', va='center',
                      fontsize=9, color=ct, fontweight='bold')
ax_c.axhline(0.5, color='white', lw=2.5)
ax_c.axhline(1.5, color='white', lw=2.0)
ax_c.axhline(4.5, color='white', lw=2.0)
plt.colorbar(imc, ax=ax_c, shrink=0.82, label='Δ MAE (%)')

# ── D: DM subperiodo ARIMA vs Chronos-2 ───────────────────────────
sub_d = dm_df[(dm_df['model1']=='arima') & (dm_df['model2']=='chronos2_C1_inst_global')]
xd = np.arange(len(PERIODS_SP))
wd = 0.18
for i, h in enumerate(HORIZONS):
    dvals, pvals = [], []
    for p in PERIODS_SP:
        row = sub_d[(sub_d['period']==p) & (sub_d['horizon']==h)]
        if len(row) and pd.notna(row['dm_stat'].values[0]):
            dvals.append(float(row['dm_stat'].values[0]))
            pvals.append(float(row['p_value'].values[0]))
        else:
            dvals.append(0.0); pvals.append(1.0)
    ax_d.bar(xd+(i-1.5)*wd, dvals, wd, label=f'h={h}',
             alpha=0.85, color=plt.cm.viridis(i/4), zorder=4)
    for xi, (dv, pv) in enumerate(zip(dvals, pvals)):
        if pv < 0.05:
            ax_d.text(xi+(i-1.5)*wd, dv+(0.18 if dv>=0 else -0.42),
                      '**', ha='center', fontsize=9, fontweight='bold')
        elif pv < 0.10:
            ax_d.text(xi+(i-1.5)*wd, dv+(0.18 if dv>=0 else -0.38),
                      '*', ha='center', fontsize=9)
ax_d.axhline(0,     color='black',   lw=1, zorder=5)
ax_d.axhline(-1.96, color='#cc0000', lw=0.9, ls='--', alpha=0.6)
ax_d.axhline( 1.96, color='#006600', lw=0.9, ls='--', alpha=0.6)
ax_d.set_xticks(xd)
ax_d.set_xticklabels([PERIOD_LABELS[p] for p in PERIODS_SP], fontsize=8.5)
ax_d.set_ylabel('Estadístico DM', fontsize=9)
ax_d.set_title('(D) DM: ARIMA vs Chronos-2 C1_inst\nDM>0 → Chronos-2 mejor  |  ** p<0.05',
               fontsize=10, fontweight='bold')
ax_d.legend(fontsize=8, ncol=2)
ax_d.grid(axis='y', alpha=0.22, zorder=0)
ax_d.set_ylim(-6.5, 6.5)
ax_d.annotate('DM > 0 → Chronos-2 mejor', xy=(0.02, 0.97), xycoords='axes fraction',
              fontsize=7, va='top', color='gray',
              bbox=dict(boxstyle='round', fc='white', alpha=0.7))

# ── Global ─────────────────────────────────────────────────────────
fig.suptitle(
    'Evaluación Comparativa — IPC Global · Rolling-origin 2021-2024\n'
    'Señales inst: Fed Funds, EPU Global, Brent · AutoARIMA: selección dinámica de órdenes · Métrica: MAE',
    fontsize=12, fontweight='bold', y=1.01)

legend_patches = [
    mpatches.Patch(color='#404040', label='Estadístico (ARIMA / ARIMAX)'),
    mpatches.Patch(color='#8B4513', label='AutoARIMA ★ (selección dinámica, −14% vs ARIMA h=12)'),
    mpatches.Patch(color='#333366', label='ARIMAX C1_inst (Fed+EPU+Brent)'),
    mpatches.Patch(color='#1a6ab0', label='Deep Learning (LSTM / N-BEATS / N-HiTS)'),
    mpatches.Patch(color='#52a852', label='Chronos-2 C1_inst ★★ — MEJOR modelo (h=3,6,12)'),
    mpatches.Patch(color='#9467bd', label='TimesFM C1_inst'),
    mpatches.Patch(color='#8fbc8f', label='TimeGPT C1_inst'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=8,
           frameon=True, bbox_to_anchor=(0.5, -0.055),
           title='Familias de modelos', title_fontsize=9)

plt.savefig(RESULTS / 'fig_MAIN_global.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('\n=== FIGURA PRINCIPAL GUARDADA: 08_results/fig_MAIN_global.png ===')

---
## 6 · Tablas de referencia completas

In [ ]:
pivot_mae  = df.pivot(index='model', columns='horizon', values='MAE').reindex(MODEL_ORDER)
pivot_mase = df.pivot(index='model', columns='horizon', values='MASE').reindex(MODEL_ORDER)
pivot_mae.columns  = [f'h={h}' for h in pivot_mae.columns]
pivot_mase.columns = [f'h={h}' for h in pivot_mase.columns]
pivot_mae.index  = [LABELS.get(m, m) for m in pivot_mae.index]
pivot_mase.index = [LABELS.get(m, m) for m in pivot_mase.index]

print('=== MAE (puntos porcentuales IPC Global) ===')
print(pivot_mae.round(4).to_string())
print()
print(f'=== MASE (escala MASE: {MASE_SCALE:.4f} | <1 = mejor que naive lag-12) ===')
print(pivot_mase.round(4).to_string())

In [ ]:
print('=' * 72)
print('SÍNTESIS DE RESULTADOS — IPC GLOBAL')
print('=' * 72)

print('\nMEJOR MODELO POR HORIZONTE (MAE):')
for h in HORIZONS:
    row = df[(df['horizon']==h) & (df['model']!='naive')].sort_values('MAE').iloc[0]
    print(f'  h={h:2d}: {LABELS.get(row["model"], row["model"]):38s} MAE={row["MAE"]:.4f}')

print('\nCOMPARACIÓN vs ARIMA (MAE):')
arima_mae = {h: metrics['arima'][f'h{h}']['MAE'] for h in HORIZONS}
aa_mae    = {h: metrics.get('auto_arima', {}).get(f'h{h}', {}).get('MAE') for h in HORIZONS}
ch_mae    = {h: metrics.get('chronos2_C1_inst_global', {}).get(f'h{h}', {}).get('MAE') for h in HORIZONS}
for h in HORIZONS:
    aa_delta = (aa_mae[h] - arima_mae[h]) / arima_mae[h] * 100 if aa_mae[h] else float('nan')
    ch_delta = (ch_mae[h] - arima_mae[h]) / arima_mae[h] * 100 if ch_mae[h] else float('nan')
    print(f'  h={h:2d}: ARIMA={arima_mae[h]:.4f}  '
          f'AutoARIMA={aa_mae[h]:.4f} ({aa_delta:+.1f}%)  '
          f'Chronos-2={ch_mae[h]:.4f} ({ch_delta:+.1f}%)')

print('\nDM SIGNIFICATIVOS (p<0.05, período completo, ARIMA vs Chronos-2):')
sub_sig = dm_df[(dm_df['model1']=='arima') & (dm_df['model2']=='chronos2_C1_inst_global')
                & (dm_df['period']=='global') & (dm_df['p_value'] < 0.05)]
if len(sub_sig):
    for _, r in sub_sig.iterrows():
        winner = 'ARIMA' if r['better']=='model1' else 'Chronos-2'
        print(f'  h={r["horizon"]}: DM={r["dm_stat"]:.3f}  p={r["p_value"]:.4f}  → {winner} mejor **')
else:
    print('  (ninguno significativo a p<0.05 en el período global)')

print('\nHALLAZGOS CLAVE:')
print('  ★  AutoARIMA (selección dinámica): mejora sobre ARIMA fijo en TODOS los horizontes')
print('     h=1: −6.3%  h=3: −13.5%  h=6: −19.1%  h=12: −13.9%  (MASE h=12=1.1343)')
print('     Nota: la selección dinámica de órdenes supera a los órdenes fijos por auto_arima inicial')
print('  ★★ Chronos-2 C1_inst: mejor modelo global — MASE h=12=0.9755 (único <1.0 en todo el estudio)')
print('     Mejora sobre AutoARIMA: h=12 = −14.3% adicional')
print('  ✓  TimesFM C1_inst: similar a AutoARIMA en h=1,3; algo peor en h=6,12')
print('  ✗  TimeGPT C1_inst: degradación relevante (+26% a +37% sobre ARIMA)')
print('  ~  Deep Learning (N-BEATS mejor): supera ARIMA a h=1, pero peor a h=12')
print('  ✗  ARIMAX C1_inst: +35% (h=1) a +7% (h=12) sobre ARIMA — exógenas no ayudan')
print('     (foundation models aprovechan mejor los covariates que ARIMAX)')
print()
print('NOTA METODOLÓGICA: Deep models con ~16 orígenes de evaluación (h=1)')
print('frente a 47 de baseline/foundation — comparación menos robusta estadísticamente.')
print('=' * 72)